# Monitoring and Observability

Set up comprehensive monitoring for the ADEPT ecosystem.

## Network Access Note (Opt-In Required)

This operator notebook requires direct access to internal services for health checks
and metrics collection. By default, JupyterLab is isolated on `frontend_net`.

To use this notebook, add JupyterLab to `application_net` via docker-compose override:

```yaml
# deployment/local/docker-compose/docker-compose.override.yaml
services:
  jupyterlab:
    networks:
      - adept_frontend_network
      - adept_application_network  # Required for direct service health checks
```

Then rebuild: `make rebuild-jupyterlab`

**Security implication**: This expands JupyterLab's network access to reach internal services.
Only enable for operator/debugging tasks. Revert when done.

**Alternative (no override needed)**: Use the nginx_proxy health endpoint for basic checks:
`https://nginx_proxy/health` — accessible from `frontend_net` without any override.

## Monitoring Stack
- Health checks and readiness probes
- Metrics collection (Prometheus)
- Log aggregation (ELK/Loki)
- Distributed tracing (Jaeger)
- Alerting strategies

## Docker Configuration
- JupyterLab service: `deployment/local/docker-compose/docker-compose.core.yaml`
- Rebuild after config changes: `make rebuild-jupyterlab`

In [ ]:
import os
import json
import httpx
import asyncio

# Load config
config = json.load(open("/tmp/adept_config.json"))
NGINX_URL = config.get('nginx_url', 'https://nginx_proxy')

async def check_service_health():
    '''Check health of all services.
    
    NOTE: Direct service access requires opt-in network configuration.
    See the markdown cell above for docker-compose.override.yaml instructions.
    
    Services on application_net (requires override):
      - agent_gateway:8081, orchestration_service:8084, gateway_registry:8086
    
    Services on frontend_net (always accessible):
      - nginx_proxy/health
    '''
    # Always-accessible health check (via nginx_proxy on frontend_net)
    frontend_services = [
        ("NGINX Proxy (frontend)", f"{NGINX_URL}/health"),
    ]
    
    # Direct service checks (requires application_net opt-in)
    internal_services = [
        ("Agent Gateway (internal)", "http://agent_gateway:8081/health"),
        ("Orchestration (internal)", "http://orchestration_service:8084/health"),
        ("Registry (internal)", "http://gateway_registry:8086/health"),
    ]
    
    async with httpx.AsyncClient(verify=False, timeout=5.0) as client:
        print("Frontend services (always accessible):")
        for name, url in frontend_services:
            try:
                response = await client.get(url)
                status = "Healthy" if response.status_code == 200 else f"Status {response.status_code}"
            except Exception as e:
                status = f"Unreachable: {str(e)[:50]}"
            print(f"  {name:35} {status}")
        
        print("\nInternal services (requires application_net opt-in):")
        for name, url in internal_services:
            try:
                response = await client.get(url)
                status = "Healthy" if response.status_code == 200 else f"Status {response.status_code}"
            except Exception as e:
                status = f"Unreachable (enable application_net in override)"
            print(f"  {name:35} {status}")

await check_service_health()

## Metrics Collection

In [ ]:
# Prometheus metrics configuration example
# These scrape targets use internal service hostnames (application_net).
# Prometheus itself runs on application_net and can reach these services.
# JupyterLab does NOT need direct access to configure Prometheus —
# this is a reference config for operator deployment.

metrics_config = '''
scrape_configs:
  - job_name: 'agent-gateway'
    static_configs:
      - targets: ['agent_gateway:8081']   # application_net
    metrics_path: '/metrics'
    
  - job_name: 'orchestration-service'
    static_configs:
      - targets: ['orchestration_service:8084']  # application_net
    metrics_path: '/metrics'

alerting:
  alertmanagers:
    - static_configs:
        - targets: ['alertmanager:9093']

rule_files:
  - 'alerts.yml'
'''

print("Prometheus scrape config (for deployment reference):")
print("Note: These internal hostnames are on application_net.")
print("Prometheus runs on that network; JupyterLab does not (by default).")
print()
print(metrics_config)

## Langfuse Observability (Sidecar Service)

ADEPT ships with Langfuse as a sidecar observability backend for LLM tracing.
All orchestration_service LLM calls are automatically traced when `OBSERVABILITY_BACKEND=langfuse`.

**Compose file**: `deployment/local/docker-compose/docker-compose.langfuse.yaml`
**Host port**: `3001` (maps to container port 3000)
**Access**: `http://localhost:3001` from your browser (not from JupyterLab container)

Key environment variables (in `.env`):
- `OBSERVABILITY_BACKEND=langfuse` — enables tracing
- `LANGFUSE_HOST=http://langfuse-web:3000` — internal hostname (application_net)
- `LANGFUSE_PUBLIC_KEY` / `LANGFUSE_SECRET_KEY` — generated by bootstrap

To access the Langfuse API from JupyterLab (requires application_net opt-in):
- `http://langfuse-web:3000/api/public/traces` — list traces
- `http://langfuse-web:3000/api/public/observations` — list observations

In [ ]:
import os
import base64

# Langfuse API access (requires application_net opt-in OR host-side access on port 3001)
LANGFUSE_HOST = os.getenv("LANGFUSE_HOST", "http://langfuse-web:3000")
LANGFUSE_PUBLIC_KEY = os.getenv("LANGFUSE_PUBLIC_KEY", "")
LANGFUSE_SECRET_KEY = os.getenv("LANGFUSE_SECRET_KEY", "")

async def query_langfuse_traces(limit: int = 5):
    """Query recent LLM traces from Langfuse.
    
    Requires application_net opt-in (langfuse-web is on application_net).
    Alternatively, access http://localhost:3001 from host browser.
    """
    if not LANGFUSE_PUBLIC_KEY or not LANGFUSE_SECRET_KEY:
        print("Langfuse credentials not available in this container.")
        print("Access Langfuse UI directly: http://localhost:3001 (from host browser)")
        print()
        print("To enable API access from JupyterLab:")
        print("  1. Add application_net to docker-compose.override.yaml")
        print("  2. Set LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY env vars")
        print("  3. Rebuild: make rebuild-jupyterlab")
        return
    
    # Basic auth: public_key:secret_key
    credentials = base64.b64encode(f"{LANGFUSE_PUBLIC_KEY}:{LANGFUSE_SECRET_KEY}".encode()).decode()
    
    async with httpx.AsyncClient(verify=False, timeout=10.0) as client:
        try:
            response = await client.get(
                f"{LANGFUSE_HOST}/api/public/traces",
                headers={"Authorization": f"Basic {credentials}"},
                params={"limit": limit, "orderBy": "timestamp.DESC"}
            )
            
            if response.status_code == 200:
                data = response.json()
                traces = data.get("data", [])
                print(f"Recent LLM Traces ({len(traces)} of {data.get('meta', {}).get('totalItems', '?')}):\n")
                for trace in traces:
                    print(f"  {trace.get('name', 'unnamed'):30} "
                          f"| {trace.get('timestamp', '')[:19]} "
                          f"| tokens: {trace.get('totalTokens', 0)}")
            elif response.status_code == 401:
                print("Authentication failed. Check LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY.")
            else:
                print(f"Langfuse API returned {response.status_code}: {response.text[:200]}")
        except httpx.ConnectError:
            print("Cannot reach langfuse-web. Enable application_net in docker-compose.override.yaml.")
            print("Or access Langfuse directly: http://localhost:3001 (from host browser)")

await query_langfuse_traces()